# Descriptive statistics and bivariate tests

This notebook reads the cleaned crash dataset produced by
[`loading_and_cleaning_data.ipynb`](loading_and_cleaning_data.ipynb), applies **the same
filters as `modeling_development.ipynb`** so that the descriptive tables and the
estimated models describe one and the same sample, and produces:

1. A summary-statistics table for one-hot-encoded categorical variables and a few
   continuous infrastructure variables.
2. A pivot table of counts and percentages by injury severity, together with
   bivariate tests (Chi-square for categorical variables; ANOVA or Kruskal-Wallis
   for continuous variables, depending on the normality test).

The two output tables are saved as CSV files and displayed inline at the end of
their respective sections so the analysis can be inspected directly in the notebook.


In [61]:
# All imports for this notebook (kept together at the top, as required).
import warnings
warnings.filterwarnings('ignore')

import re

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, kruskal, kstest

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)


## 1. Load and filter the dataset

The three filters below are those of `modeling_development.ipynb`, applied in the
same order, so that the two notebooks describe the same population:

1. `severity != -1` -- the severity is unknown;
2. `Vehicle` in E-bike / Bike / E-PMD / **Pedestrian** -- pedestrians are part of
   the estimation sample, they are the third party of one of the four segments;
3. `dropna(['age', 'severity'])` -- the age is needed by every model;
4. the union of the four estimation segments, which removes the 200 rows that
   belong to no model: 199 with `catu == 3` -- pedestrians, excluded by the
   `catu` filter of the car, MMV and single-vehicle models -- and one fatal
   micro-mobility crash removed by the `severity != 3` filter of the MMV model.

The tables therefore describe **exactly** the 15 386 observations the four models
are estimated on.


In [62]:

dataset = pd.read_csv('final_processed_crash_dataset.csv', low_memory=False)

dataset = dataset.loc[dataset['severity'] != -1]
dataset = dataset.loc[dataset['Vehicle'].isin(['E-PMD', 'Bike', 'E-bike', 'Pedestrian'])]

dataset = dataset.dropna(subset=['age', 'severity'])
second_party = dataset['vehicle_type_2']
is_driver_or_passenger = dataset['catu'].isin([1, 2])

is_car = second_party.isin(['Cars', 'Large motorized vehicle',
                            'Light motorized vehicle']) & is_driver_or_passenger
is_mmv = ((second_party == 'Micromobility vehicle') & is_driver_or_passenger)
is_sv = (second_party == 'No other vehicle') & is_driver_or_passenger
is_pedestrian = dataset['Num_Acc'].isin(
    dataset.loc[second_party == 'Pedestrian', 'Num_Acc']
)

dataset = dataset.loc[is_car | is_mmv | is_pedestrian | is_sv]

# Convenience alias used in the bivariate tables further down.
dataset['sev'] = dataset['severity']

print(f'Rows after filtering: {len(dataset):,}')   # doit valoir 15 386
print(f'Unique accidents:     {dataset["Num_Acc"].nunique():,}')
dataset.head()


Rows after filtering: 15,387
Unique accidents:     13,024


,Unnamed: 0,Num_Acc,day,month,Year,hrmn,lum,com,int,atm,col,adr,lat,long,geometry,catr,circ,nbv,vosp,prof,plan,surf,infra,situ,vma,...,Gender_driver,severity_driver,Maneuver,Obstacle,Vehicle type,danger_rank,age_2,vehicle_type_2,Vehicle_2,Maneuver_2,Gender_2,severity_2,Point of impact_2,id_vehicule_opposite_2,Age category involved,age_3,vehicle_type_3,Vehicle_3,Maneuver_3,Gender_3,severity_3,Point of impact_3,id_vehicule_opposite_3,age_opposite_mean,sev
1,1,2.022000e+11,21,10,2022,16:32,1,75106,1,1,3,RUE DE VAUGIRARD,48.847999,2.330176,POINT (2.330176 48.847999),4,2,2,0,1,1,1,0,1,30,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,62.0,Cars,Car,Door openied,Female,1.0,Front,813 926,61+,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.0,2
3,3,2.022000e+11,20,10,2022,13:00,1,75105,2,1,3,BOULEVARD SAINT GERMAIN,48.851387,2.343186,POINT (2.343186 48.851387),4,1,3,1,1,1,1,0,5,30,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,20.0,Cars,Car,Turning right,Male,1.0,Front,813 924,21-40,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,2
4,4,2.022000e+11,21,10,2022,11:25,1,75113,4,1,6,BOULEVARD KELLERMANN,48.821028,2.354515,POINT (2.354515 48.821028),4,2,6,1,2,2,2,0,5,50,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,999.0,No other vehicle,NaN,Missing,NaN,NaN,NaN,NaN,NaN,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2
7,7,2.022000e+11,21,10,2022,19:40,5,93049,2,1,3,Avenue Jean Jaurès / Avenue Victor Hugo,48.858700,2.506540,POINT (2.50654 48.8587),4,1,2,0,1,1,1,0,1,30,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,48.0,Cars,Car,Without change of direction,Female,1.0,Front,813 884,41-60,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.0,2
9,9,2.022000e+11,21,10,2022,20:00,5,92048,1,1,2,JEAN JAURES (AVENUE) N° 8 A 72,48.812880,2.246200,POINT (2.2462 48.81288),4,2,2,0,4,1,1,0,1,50,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,62.0,Cars,Car,Other,Male,1.0,Back,813 866,61+,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.0,2


## 2. Variable groups

There is a single unit of observation: **one row per person involved**. Every
variable, including those describing the crash environment (lighting, weather,
road type, ...), is counted once per individual rather than once per accident.

A crash involving two people therefore contributes its road type twice. This is
the unit the models are estimated on, so the descriptive tables and the
estimation samples describe the same thing -- at the cost of over-weighting the
environment of crashes involving more people.

`var_cont` lists the continuous variables analysed in the bivariate section.


In [63]:
# Familles de variables. Elles ne servent plus a changer d'unite d'observation
# -- tout est compte par usager -- mais restent utiles pour documenter d'ou
# vient chaque variable.

# Propre a l'usager observe : une ligne = une personne.
RIDER_VARS = {
    'age', 'Age category', 'Gender', 'Vehicle', 'User category', 'Helmet',
    'Reflective jacket', 'Trip purpose', 'Number of passengers',
    'Maneuver', 'Point of impact',
}

# Propre aux tiers : lues en « au moins un tiers presente cette
# caracteristique », en combinant les colonnes `_2` (premier tiers) et `_3`
# (second). Voir SECOND_PARTY_GROUPS en section 3.
SECOND_PARTY_VARS = {
    'vehicle_type_2', 'vehicle_type_3', 'Vehicle_2', 'Vehicle_3',
    'Maneuver_2', 'Maneuver_3', 'Gender_2', 'Gender_3',
    'Point of impact_2', 'Point of impact_3',
    'age_2', 'age_opposite_mean', 'Age category involved',
}

# Continuous variables analysed in the bivariate section.
var_cont = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'surfacechaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]

## 3. Summary statistics for one-hot-encoded categorical and continuous variables

We one-hot-encode a small set of categorical variables, merge in the continuous
infrastructure variables from the accident-level frame, and compute mean / median /
min / max for each column. The resulting table is saved as
`summary_statistics_categorical_continuous.csv` and is displayed below.


In [64]:
var_added = pd.get_dummies(
    dataset[['Num_Acc', 'Cycle facilities', 'Pavement',
             'Crossroad', 'positionnement_piste',
             'Road type', 'Reglementation']]
)
var_added = var_added.astype(int)  # boolean -> integer

var_added = var_added.merge(
    dataset[['Num_Acc', 'surfacechaussee', 'pentemoyenne',
             'largeurtrottoirdroit']].drop_duplicates('Num_Acc'),
    on='Num_Acc',
    how='left'
)

results_df = pd.DataFrame([
    {
        'Column': col,
        'Mean':   var_added[col].mean(),
        'Median': var_added[col].median(),
        'Min':    var_added[col].min(),
        'Max':    var_added[col].max(),
    }
    for col in var_added.columns
])

results_df.to_csv('summary_statistics_categorical_continuous.csv', index=False)
results_df


,Column,Mean,Median,Min,Max
0,Num_Acc,2.021233e+11,2.021000e+11,2.019000e+11,2.023001e+11
1,Cycle facilities_Bus lane,1.197764e-01,0.000000e+00,0.000000e+00,1.000000e+00
2,Cycle facilities_Cycle lane,1.464223e-01,0.000000e+00,0.000000e+00,1.000000e+00
3,Cycle facilities_Cycle path/Greenway,1.767726e-01,0.000000e+00,0.000000e+00,1.000000e+00
4,Cycle facilities_No cycle facilities,5.257035e-01,1.000000e+00,0.000000e+00,1.000000e+00
5,Cycle facilities_Pedestrianized street,3.132514e-02,0.000000e+00,0.000000e+00,1.000000e+00
6,Pavement_Asphalt,7.527783e-01,1.000000e+00,0.000000e+00,1.000000e+00
7,Pavement_Concrete,8.903620e-03,0.000000e+00,0.000000e+00,1.000000e+00
8,Pavement_Other,1.828167e-01,0.000000e+00,0.000000e+00,1.000000e+00
9,Pavement_Paved,5.550140e-02,0.000000e+00,0.000000e+00,1.000000e+00


## 4. Bivariate analysis (counts, percentages, p-values)

For each categorical variable we report counts and percentages by severity level,
and a Chi-square p-value per category, left as NaN when Cochran's rule is
violated -- that is, when an expected frequency falls below 1, or when more than
20% of them fall below 5. Applying the rule as Cochran stated it rather than in
its simplified "every expected frequency above 5" form makes 119 of the 133
possible tests reportable instead of 80. For each continuous variable we report the median and
interquartile range by severity, and an ANOVA p-value when the groups are normal
(Kolmogorov-Smirnov test) or a Kruskal-Wallis p-value otherwise.


In [65]:
COCHRAN_MINIMUM = 1          # aucune case attendue en dessous de 1
COCHRAN_SMALL = 5            # une case est « petite » sous 5
COCHRAN_SHARE = 0.20         # au plus 20 % de petites cases


def cochran_is_satisfied(expected):

    expected = np.asarray(expected, dtype=float)
    if expected.min() < COCHRAN_MINIMUM:
        return False
    return (expected < COCHRAN_SMALL).mean() <= COCHRAN_SHARE


def calculate_chi2_p_value_for_each_category(data, categorical_var):
    """Per-category Chi-square p-value, NaN if Cochran's rule is violated."""
    p_values = {}
    for category in data[categorical_var].unique():
        contingency = pd.crosstab(data['severity'], data[categorical_var] == category)
        chi2, p, _, expected = chi2_contingency(contingency)
        if not cochran_is_satisfied(expected):
            p_values[category] = np.nan
        else:
            p_values[category] = '<0.001' if p < 0.001 else str(round(p, 3))
    return p_values


def count_by_severity(data, categorical_var):
    """Counts and percentages of `categorical_var` within each severity level."""
    counts = (data
              .groupby(['severity', categorical_var])
              .size()
              .astype(int)
              .reset_index(name='count'))
    total_counts = counts.groupby(categorical_var)['count'].transform('sum')
    counts['percentage'] = (counts['count'] / total_counts * 100).round(2)
    counts['count_percentage'] = counts.apply(
        lambda row: f"{row['count']} ({row['percentage']}%)", axis=1
    )
    return counts


def compare_continuous_variable(data, group_var, continuous_var):
    """ANOVA if all groups are normal (KS test, alpha=0.05), Kruskal-Wallis otherwise."""
    groups_data = [
        data[data[group_var] == g][continuous_var].dropna().values
        for g in data[group_var].unique()
    ]
    normal = all(
        kstest(g, 'norm', args=(g.mean(), g.std())).pvalue > 0.05
        for g in groups_data
    )
    _, p_value = (f_oneway(*groups_data) if normal else kruskal(*groups_data))
    return '<0.001' if p_value < 0.001 else str(round(p_value, 3))


# --- Caracteristiques du tiers : « au moins un » ------------------------------
# Un usager peut faire face a deux tiers (`_2` et `_3`). Compter une modalite par
# tiers gonflerait les effectifs et donnerait 2 la ou les deux tiers partagent la
# caracteristique : on lit donc « au moins un tiers presente cette
# caracteristique ». Consequence : les pourcentages d'une meme variable ne
# somment plus a 100 %.
SECOND_PARTY_GROUPS = {
    'vehicle_type_2': ['vehicle_type_2', 'vehicle_type_3'],
    'Vehicle_2': ['Vehicle_2', 'Vehicle_3'],
    'Maneuver_2': ['Maneuver_2', 'Maneuver_3'],
    'Gender_2': ['Gender_2', 'Gender_3'],
    'Point of impact_2': ['Point of impact_2', 'Point of impact_3'],
}

SEVERITY_LEVELS = [1, 2, 3]


def any_party_levels(data, columns):
    """Modalites presentes chez l'un ou l'autre des tiers."""
    levels = set()
    for column in columns:
        if column in data.columns:
            levels.update(data[column].dropna().unique())
    return sorted(levels, key=str)


def any_party_mask(data, columns, level):
    """Individus dont AU MOINS un tiers presente la modalite."""
    mask = pd.Series(False, index=data.index)
    for column in columns:
        if column in data.columns:
            mask |= (data[column] == level)
    return mask


def chi2_p_value(severity, mask):
    """Chi-2 de la modalite contre la severite ; None si Cochran est violee."""
    contingency = pd.crosstab(severity, mask)
    if contingency.shape[1] < 2 or contingency.shape[0] < 2:
        return None
    _, p_value, _, expected = chi2_contingency(contingency)
    if not cochran_is_satisfied(expected):
        return None
    return '<0.001' if p_value < 0.001 else str(round(p_value, 3))


def any_party_block(sample, variable, columns):
    """Bloc « au moins un tiers » : effectifs par severite et test par modalite."""
    records = []
    for level in any_party_levels(sample, columns):
        mask = any_party_mask(sample, columns, level)
        total = int(mask.sum())
        if not total:
            continue
        counts = sample.loc[mask, 'severity'].value_counts()
        record = {'Variable': variable, 'Category': level}
        for severity in SEVERITY_LEVELS:
            count = int(counts.get(severity, 0))
            record[severity] = f'{count} ({round(count / total * 100, 2)}%)'
        record['p_value'] = chi2_p_value(sample['severity'], mask)
        records.append(record)
    return pd.DataFrame.from_records(records) if records else None

In [66]:
categorical_vars = [
    'Age category', 'Gender', 'Vehicle', 'User category', 'Helmet', 'sev',
    'Point of impact_2', 'Reflective jacket', 'Lighting conditions',
    'Weather conditions', 'Point of impact', 'Maneuver', 'Cycle facilities',
    'Accident location', 'Trip purpose', 'Max speed', 'Intersection',
    'Crossroad', 'Long profile', 'Pavement', 'Surface condition', 'Road width',
    'vehicle_type_2', 'Maneuver_2', 'Gender_2', 'Age category involved',
    'positionnement_piste', 'Road type'
]

continuous_vars = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'largeurchaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]

pivot_results = []

# Categorical variables: counts/percentages and Chi-square per category.
for var in categorical_vars:
    if var in SECOND_PARTY_GROUPS:
        block = any_party_block(dataset, var, SECOND_PARTY_GROUPS[var])
        if block is not None:
            pivot_results.append(block)
        continue
    counts_df = count_by_severity(dataset, var)
    pivot_df = counts_df.pivot(index=var, columns='severity',
                               values='count_percentage').reset_index()
    pivot_df['Variable'] = var
    pivot_df['Category'] = pivot_df[var]
    pivot_df.drop(columns=var, inplace=True)
    p_values = calculate_chi2_p_value_for_each_category(dataset, var)
    pivot_df['p_value'] = pivot_df['Category'].map(p_values)
    pivot_results.append(pivot_df)

# Continuous variables: median [Q1-Q3] and ANOVA / Kruskal-Wallis p-value.
for var in continuous_vars:
    df = dataset[['severity', var]].dropna(subset=[var])
    df = df[df[var] != 999]  # 999 is the Biogeme missing-data sentinel
    stats_df = df.groupby('severity')[var].agg(
        median='median',
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75)
    ).reset_index()
    stats_df['median_iqr'] = stats_df.apply(
        lambda row: f"{row['median']:.2f} [{row['q1']:.2f}-{row['q3']:.2f}]", axis=1
    )
    stats_df = stats_df[['severity', 'median_iqr']].set_index('severity').T
    stats_df.columns = [1, 2, 3]
    stats_df['Variable'] = var
    stats_df['p_value'] = compare_continuous_variable(df, 'severity', var)
    pivot_results.append(stats_df)

final_pivot_df = pd.concat(pivot_results, ignore_index=True)
cols_order = ['Variable', 'Category'] + [c for c in final_pivot_df.columns
                                          if c not in ['Variable', 'Category']]
final_pivot_df = final_pivot_df[cols_order]

# Rename a few raw column names for the printed/exported table.
final_pivot_df = final_pivot_df.replace({
    'vehicle_type_2': 'Third-party vehicle type',
    'Maneuver_2': 'Third-party maneuver',
    'Gender_2': 'Third-party gender',
    'Point of impact_opposite': 'Third-party impact location',
    'vma': 'Speed limit',
    'surfacechaussee': 'Road surface width',
    'pentemoyenne': 'Average slope',
    'largeurtrottoirdroit': 'Sidewalk width',
    'age': 'Individual age',
    'number of involved vehicles': 'Number of vehicles involved',
})

print(f'Rows in the final pivot table: {len(final_pivot_df)}')
final_pivot_df


Rows in the final pivot table: 141


,Variable,Category,1,2,3,p_value
0,Age category,0-20,344 (16.87%),1687 (82.74%),8 (0.39%),<0.001
1,Age category,21-40,1163 (15.25%),6428 (84.28%),36 (0.47%),<0.001
2,Age category,41-60,398 (9.58%),3728 (89.7%),30 (0.72%),<0.001
3,Age category,61+,75 (4.8%),1464 (93.67%),24 (1.54%),<0.001
4,Gender,Female,364 (7.22%),4652 (92.26%),26 (0.52%),<0.001
5,Gender,Male,1617 (15.63%),8656 (83.67%),72 (0.7%),<0.001
6,Vehicle,Bike,1115 (12.56%),7708 (86.82%),55 (0.62%),0.37
7,Vehicle,E-PMD,569 (14.24%),3402 (85.14%),25 (0.63%),0.011
8,Vehicle,E-bike,160 (14.39%),941 (84.62%),11 (0.99%),0.084
9,Vehicle,Pedestrian,137 (9.78%),1257 (89.72%),7 (0.5%),0.001
